In [3]:
import torch
import torch.nn as nn
import torchvision.models as models
import wandb
import os
from datasets import load_dataset, load_from_disk
from PIL import ImageFile
from torchvision import transforms
from torch.utils.data import DataLoader
from datetime import datetime

ImageFile.LOAD_TRUNCATED_IMAGES = True


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
ds = load_dataset("food101")

save_dir = "/content/drive/MyDrive/food101_data"
ds.save_to_disk(save_dir)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00008.parquet:   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00008.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

data/train-00002-of-00008.parquet:   0%|          | 0.00/472M [00:00<?, ?B/s]

data/train-00003-of-00008.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

...

data/validation-00002-of-00003.parquet:   0%|          | 0.00/426M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/75750 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/25250 [00:00<?, ? examples/s]

Saving the dataset (0/8 shards):   0%|          | 0/75750 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/25250 [00:00<?, ? examples/s]

In [6]:
PROJECT_NAME = "CS7643-Deep Learning"
machine = "cpu"

if torch.cuda.is_available():
    machine = "cuda"
elif torch.mps.is_available():
    machine = "mps"


device = torch.device(machine)

print("using device ", device)

using device  cuda


In [10]:
wandb.login()

os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rajeman (rajeman-emm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [11]:
ds = load_from_disk(save_dir)

train_ds = ds["train"]

image_size = 224

transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [12]:
BATCH_SIZE = 128


class Food101(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.ds = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        image = item["image"]
        label = item["label"]
        if self.transform:
            image = self.transform(image)
        return image, label


train_dataset = Food101(train_ds, transform=transform)
split = ds["validation"].train_test_split(test_size=0.5, seed=42)
val_dataset = Food101(split["train"], transform=transform)
test_dataset = Food101(split["test"], transform=transform)

_dl_kwargs = {"batch_size": BATCH_SIZE, "pin_memory": device.type == "cuda"}
train_loader = DataLoader(train_dataset, shuffle=True, **_dl_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **_dl_kwargs)

num_classes = 101

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 203MB/s]


In [13]:
for param in model.parameters():
    param.requires_grad = False

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

model = model.to(device)

In [14]:
lr = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=lr)


def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)




def evaluate(model, loader):
    model.eval()

    correct_top1 = 0
    correct_top5 = 0
    total = 0
    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, top5 = outputs.topk(5, dim=1)
            _, top1 = outputs.topk(1, dim=1)

            total += labels.size(0)

            correct_top1 += (top1.squeeze() == labels).sum().item()

            for i in range(labels.size(0)):
                if labels[i] in top5[i]:
                    correct_top5 += 1

    return {
        "top1_acc": correct_top1 / total,
        "top5_acc": correct_top5 / total,
        "val_loss": total_loss / len(loader)
    }



In [ ]:

epochs = 10


RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
run = wandb.init(project=PROJECT_NAME, name=RUN_NAME)

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader)
    metrics = evaluate(model, val_loader)

    run.log({
        "train_loss": train_loss,
        "top1_acc": metrics["top1_acc"],
        "top5_acc": metrics["top5_acc"],
        "val_loss": metrics["val_loss"],
        "lr": lr,
    }, step=epoch)


    print(f"Epoch {epoch}")
    print("Train Loss:", train_loss)
    print(metrics)

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 0
Train Loss: 2.4675892410246103
{'top1_acc': 0.5565148514851486, 'top5_acc': 0.8118019801980199, 'val_loss': 1.7514308666942096}
Epoch 1
Train Loss: 1.8564252128472198
{'top1_acc': 0.5837623762376237, 'top5_acc': 0.8308118811881188, 'val_loss': 1.595668952874463}
Epoch 2
Train Loss: 1.7311704301753559
{'top1_acc': 0.5868514851485148, 'top5_acc': 0.8337425742574257, 'val_loss': 1.577013904398138}
Epoch 3
Train Loss: 1.6629552297495507
{'top1_acc': 0.599049504950495, 'top5_acc': 0.8371485148514851, 'val_loss': 1.5405701892544525}
Epoch 4
Train Loss: 1.6081098634246234
{'top1_acc': 0.6036435643564356, 'top5_acc': 0.843009900990099, 'val_loss': 1.5165328678458627}
Epoch 5
Train Loss: 1.563142674798901
{'top1_acc': 0.6095841584158416, 'top5_acc': 0.850059405940594, 'val_loss': 1.4831018387669264}
Epoch 6
Train Loss: 1.5235085503475085
{'top1_acc': 0.6086336633663366, 'top5_acc': 0.8461782178217822, 'val_loss': 1.4985782498061055}


In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=device.type == "cuda",
)

test_metrics = evaluate(model, test_loader)

print("\nFinal Test Results:")
print(test_metrics)

run.log({
    "final_test_loss": test_metrics["val_loss"],
    "final_test_top1_acc": test_metrics["top1_acc"],
    "final_test_top5_acc": test_metrics["top5_acc"],
})

In [ ]:
save_path = "/content/drive/MyDrive/resnet50_food101-" + RUN_NAME +  ".pth"
torch.save(model.state_dict(), save_path)